# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
- https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Name:", metadata.name)
print("Description:", metadata.description)
print("Published:", metadata.datePublished)
print("Version:", metadata.version)
print("License:", metadata.license)
print("Identifier:", metadata.identifier)


## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

Below, you'll see all available record sets and their associated fields, each referenced by their `@id`.

In [ ]:
# List available record sets with their @id
record_sets = list(dataset.record_sets())
print("Available Record Sets:")
for rs in record_sets:
    print(f"- RecordSet: {rs['@id']} (name: {rs.get('name', 'Unnamed')})")
    print("  Fields:")
    for field in dataset.fields(record_set=rs['@id']):
        print(f"    - Field @id: {field['@id']} (name: {field.get('name', 'Unnamed')}, dataType: {field.get('dataType', 'Unknown')})")

# For further exploration, let's save the first record set @id
if record_sets:
    record_set_id = record_sets[0]['@id']
else:
    record_set_id = None

### Preview Records by `@id`
Let's examine a few records from the primary record set, referencing exclusively by the record set `@id`.

In [ ]:
# Print first 3 records from the primary RecordSet using its @id
if record_set_id is not None:
    print(f"\nSample records from RecordSet @id: {record_set_id}")
    for idx, record in enumerate(dataset.records(record_set=record_set_id)):
        if idx >= 3:
            break
        print(record)
else:
    print("No record sets found in the dataset.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All record sets and fields are referenced by their `@id`.

Let's collect data from every available record set for exploratory data analysis.

In [ ]:
# Extract all data from available record sets, referenced by their @id
dataframes = {}
record_sets_ids = [rs['@id'] for rs in record_sets]

for rid in record_sets_ids:
    print(f"Loading records for RecordSet @id: {rid}")
    records = list(dataset.records(record_set=rid))
    df = pd.DataFrame(records)
    dataframes[rid] = df
    print("Columns (@id):", df.columns.tolist())

if record_sets_ids:
    # Preview the first dataframe
    print(f"\nPreview of DataFrame for RecordSet @id: {record_sets_ids[0]}")
    display(dataframes[record_sets_ids[0]].head())
else:
    print("No record sets to extract data from.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

All manipulations reference columns and fields by their `@id`.

In [ ]:
# Find numeric fields for analysis from the first record set
if record_set_id is not None and record_set_id in dataframes:
    df = dataframes[record_set_id]
    # Extract field definitions
    numeric_fields = []
    for f in dataset.fields(record_set=record_set_id):
        if f.get('dataType') in ['schema:Integer', 'schema:Float', 'schema:Number']:
            if f['@id'] in df.columns:
                numeric_fields.append(f['@id'])

    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field @id: {numeric_field_id} for EDA")
        # Choose a threshold for filtering
        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Find a categorical (string) field to group by
        group_fields = [f['@id'] for f in dataset.fields(record_set=record_set_id) if f.get('dataType') == 'schema:Text' and f['@id'] in df.columns]
        if group_fields:
            group_field_id = group_fields[0]
            print(f"Grouped data by {group_field_id}:")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric fields found for EDA.")
else:
    print("No dataframe loaded for main RecordSet.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

All plots reference fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id is not None and record_set_id in dataframes and numeric_fields:
    df = dataframes[record_set_id]
    numeric_field_id = numeric_fields[0]

    plt.figure(figsize=(8,6))
    sns.histplot(df[numeric_field_id], kde=True, bins=20)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If group_field_id present, show boxplot
    if 'group_field_id' in locals():
        plt.figure(figsize=(8,6))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides comprehensive clinicopathological details on cancer survivors with second primary colorectal cancer, referenced via Croissant schema `@id` fields.
- Using the `mlcroissant` library, we loaded metadata and identified record sets, extracted records by `@id`, and applied EDA like filtering, normalization, and grouping.
- We visualized the distribution of numeric clinicopathological features, which can support further clinical stratification and modeling.
- All steps and references are based on the Croissant schema standards for reproducibility and interoperability.